In [7]:
!pip install --upgrade gradio spacy nltk pandas matplotlib
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [8]:
import gradio as gr
import spacy
import nltk
import pandas as pd
import matplotlib.pyplot as plt

import base64
import io
import html

from nltk.stem import PorterStemmer
from spacy import displacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Initialize stemmer
stemmer = PorterStemmer()

print("NLP model loaded successfully!")

NLP model loaded successfully!


In [9]:
def analyze_text(text, operation):

    # -------------------------------------------------
    # CHECK INPUT
    # -------------------------------------------------

    if not text or not text.strip():
        return """
        <div style="
            padding:20px;
            background:#fff3cd;
            color:#664d03;
            border-radius:10px;
            border:1px solid #ffecb5;
            font-family:Arial,sans-serif;
        ">
            ⚠️ Please enter some text first.
        </div>
        """

    # Process text with spaCy
    doc = nlp(text)

    # =================================================
    # 1. NAMED-ENTITY RELATIONSHIP
    # =================================================

    if operation == "Named-Entity Relationship":

        rows = ""

        for ent in doc.ents:

            rows += f"""
            <tr>
                <td>{html.escape(ent.text)}</td>
                <td>{html.escape(ent.label_)}</td>
                <td>{html.escape(spacy.explain(ent.label_) or "No description available")}</td>
            </tr>
            """

        if not rows:
            rows = """
            <tr>
                <td colspan="3" style="text-align:center;">
                    No named entities detected.
                </td>
            </tr>
            """

        return f"""
        <div style="
            background:#ffffff;
            color:#111111;
            padding:25px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="color:#111111; margin-top:0;">
                🏷️ Named Entities
            </h3>

            <table style="
                width:100%;
                border-collapse:collapse;
                margin-top:15px;
            ">

                <tr style="
                    background:#4f46e5;
                    color:white;
                ">
                    <th style="padding:12px; text-align:left;">
                        Entity
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Label
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Description
                    </th>
                </tr>

                {rows}

            </table>

        </div>
        """

    # =================================================
    # 2. POS TAGGING
    # =================================================

    elif operation == "POS Tagging":

        rows = ""

        for token in doc:

            if not token.is_space:

                rows += f"""
                <tr>
                    <td>{html.escape(token.text)}</td>
                    <td>{html.escape(token.pos_)}</td>
                    <td>{html.escape(token.tag_)}</td>
                    <td>{html.escape(token.lemma_)}</td>
                </tr>
                """

        return f"""
        <div style="
            background:#ffffff;
            color:#111111;
            padding:25px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="color:#111111; margin-top:0;">
                🧩 Part-of-Speech Tagging
            </h3>

            <table style="
                width:100%;
                border-collapse:collapse;
            ">

                <tr style="
                    background:#4f46e5;
                    color:white;
                ">

                    <th style="padding:12px; text-align:left;">
                        Word
                    </th>

                    <th style="padding:12px; text-align:left;">
                        POS
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Detailed Tag
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Lemma
                    </th>

                </tr>

                {rows}

            </table>

        </div>
        """

    # =================================================
    # 3. POS DISTRIBUTION
    # =================================================

    elif operation == "POS Distribution":

        pos_counts = {}

        for token in doc:

            if not token.is_space:

                pos = token.pos_

                pos_counts[pos] = pos_counts.get(pos, 0) + 1

        pos_df = pd.DataFrame(
            list(pos_counts.items()),
            columns=["POS", "Count"]
        )

        # Create chart
        fig, ax = plt.subplots(figsize=(9, 4.5))

        ax.bar(
            pos_df["POS"],
            pos_df["Count"]
        )

        ax.set_title("POS Tag Distribution")
        ax.set_xlabel("Part of Speech")
        ax.set_ylabel("Frequency")

        plt.xticks(rotation=30)
        plt.tight_layout()

        # Convert chart to Base64
        buffer = io.BytesIO()

        fig.savefig(
            buffer,
            format="png",
            dpi=150,
            bbox_inches="tight"
        )

        plt.close(fig)

        buffer.seek(0)

        chart_base64 = base64.b64encode(
            buffer.read()
        ).decode("utf-8")

        # Table rows
        rows = ""

        for _, row in pos_df.iterrows():

            rows += f"""
            <tr>
                <td>{html.escape(str(row["POS"]))}</td>
                <td>{int(row["Count"])}</td>
            </tr>
            """

        return f"""
        <div style="
            background:#ffffff;
            color:#111111;
            padding:25px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="color:#111111; margin-top:0;">
                📊 POS Distribution
            </h3>

            <table style="
                width:100%;
                border-collapse:collapse;
                margin-bottom:25px;
            ">

                <tr style="
                    background:#4f46e5;
                    color:white;
                ">

                    <th style="padding:12px; text-align:left;">
                        POS
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Count
                    </th>

                </tr>

                {rows}

            </table>

            <div style="text-align:center;">

                <img
                    src="data:image/png;base64,{chart_base64}"
                    style="
                        width:100%;
                        max-width:900px;
                        height:auto;
                    "
                >

            </div>

        </div>
        """

    # =================================================
    # 4. LEMMATIZATION
    # =================================================

    elif operation == "Lemmatization":

        rows = ""

        for token in doc:

            if not token.is_space:

                rows += f"""
                <tr>

                    <td>{html.escape(token.text)}</td>

                    <td>{html.escape(token.lemma_)}</td>

                    <td>{html.escape(token.pos_)}</td>

                </tr>
                """

        return f"""
        <div style="
            background:#ffffff;
            color:#111111;
            padding:25px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="color:#111111; margin-top:0;">
                🔤 Lemmatization
            </h3>

            <table style="
                width:100%;
                border-collapse:collapse;
            ">

                <tr style="
                    background:#4f46e5;
                    color:white;
                ">

                    <th style="padding:12px; text-align:left;">
                        Original Word
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Lemma
                    </th>

                    <th style="padding:12px; text-align:left;">
                        POS
                    </th>

                </tr>

                {rows}

            </table>

        </div>
        """

    # =================================================
    # 5. STEMMING
    # =================================================

    elif operation == "Stemming":

        rows = ""

        for token in doc:

            if not token.is_space and token.is_alpha:

                stem = stemmer.stem(token.text)

                rows += f"""
                <tr>

                    <td>{html.escape(token.text)}</td>

                    <td>{html.escape(stem)}</td>

                </tr>
                """

        return f"""
        <div style="
            background:#ffffff;
            color:#111111;
            padding:25px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="color:#111111; margin-top:0;">
                ✂️ Stemming
            </h3>

            <table style="
                width:100%;
                border-collapse:collapse;
            ">

                <tr style="
                    background:#4f46e5;
                    color:white;
                ">

                    <th style="padding:12px; text-align:left;">
                        Original Word
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Stem
                    </th>

                </tr>

                {rows}

            </table>

        </div>
        """

    # =================================================
    # 6. MORPHOLOGY
    # =================================================

    elif operation == "Morphology":

        rows = ""

        for token in doc:

            if not token.is_space:

                morphology = str(token.morph)

                if not morphology:
                    morphology = "—"

                rows += f"""
                <tr>

                    <td>{html.escape(token.text)}</td>

                    <td>{html.escape(token.lemma_)}</td>

                    <td>{html.escape(token.pos_)}</td>

                    <td>{html.escape(morphology)}</td>

                </tr>
                """

        return f"""
        <div style="
            background:#ffffff;
            color:#111111;
            padding:25px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="color:#111111; margin-top:0;">
                🧬 Morphological Analysis
            </h3>

            <table style="
                width:100%;
                border-collapse:collapse;
            ">

                <tr style="
                    background:#4f46e5;
                    color:white;
                ">

                    <th style="padding:12px; text-align:left;">
                        Word
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Lemma
                    </th>

                    <th style="padding:12px; text-align:left;">
                        POS
                    </th>

                    <th style="padding:12px; text-align:left;">
                        Morphology
                    </th>

                </tr>

                {rows}

            </table>

        </div>
        """

    # =================================================
    # 7. DEPENDENCIES
    # =================================================

    elif operation == "Dependencies":

        # Generate dependency visualization
        svg = displacy.render(
            doc,
            style="dep",
            jupyter=False,
            options={
                "distance": 120,
                "compact": False
            }
        )

        # Make SVG readable
        svg = svg.replace(
            "</svg>",
            """
            <style>
                text {
                    fill: #111111 !important;
                    font-family: Arial, sans-serif !important;
                }

                path {
                    stroke: #222222 !important;
                }

                marker path {
                    fill: #222222 !important;
                    stroke: #222222 !important;
                }
            </style>
            </svg>
            """
        )

        # Convert SVG to Base64
        svg_base64 = base64.b64encode(
            svg.encode("utf-8")
        ).decode("utf-8")

        # Dependency table
        dependency_rows = ""

        for token in doc:

            if not token.is_space:

                dependency_rows += f"""
                <div style="
                    display:grid;
                    grid-template-columns:1fr 1fr 1.5fr 1fr;
                    background:#ffffff !important;
                    color:#111111 !important;
                    border-bottom:1px solid #dddddd;
                ">

                    <div style="
                        padding:12px;
                        color:#111111 !important;
                        background:#ffffff !important;
                    ">
                        {html.escape(token.text)}
                    </div>

                    <div style="
                        padding:12px;
                        color:#111111 !important;
                        background:#ffffff !important;
                    ">
                        {html.escape(token.pos_)}
                    </div>

                    <div style="
                        padding:12px;
                        color:#111111 !important;
                        background:#ffffff !important;
                    ">
                        {html.escape(token.dep_)}
                    </div>

                    <div style="
                        padding:12px;
                        color:#111111 !important;
                        background:#ffffff !important;
                    ">
                        {html.escape(token.head.text)}
                    </div>

                </div>
                """

        return f"""
        <div style="
            background:#ffffff !important;
            color:#111111 !important;
            padding:30px;
            border-radius:12px;
            width:100%;
            box-sizing:border-box;
            font-family:Arial,sans-serif;
        ">

            <h3 style="
                color:#111111 !important;
                margin:0 0 20px 0;
            ">
                🔗 Dependency Tree
            </h3>

            <div style="
                background:#ffffff !important;
                border:1px solid #dddddd;
                border-radius:10px;
                padding:20px;
                text-align:center;
                overflow-x:auto;
            ">

                <img
                    src="data:image/svg+xml;base64,{svg_base64}"
                    style="
                        width:100%;
                        max-width:1100px;
                        height:auto;
                        display:block;
                        margin:auto;
                    "
                >

            </div>

            <h3 style="
                color:#111111 !important;
                margin:30px 0 15px 0;
            ">
                📋 Dependency Information
            </h3>

            <div style="
                width:100%;
                border:1px solid #dddddd;
                border-radius:8px;
                overflow:hidden;
                background:#ffffff !important;
                color:#111111 !important;
            ">

                <!-- HEADER -->

                <div style="
                    display:grid;
                    grid-template-columns:1fr 1fr 1.5fr 1fr;
                    background:#4f46e5 !important;
                    color:#ffffff !important;
                    font-weight:bold;
                ">

                    <div style="
                        padding:12px;
                        background:#4f46e5 !important;
                        color:#ffffff !important;
                    ">
                        Word
                    </div>

                    <div style="
                        padding:12px;
                        background:#4f46e5 !important;
                        color:#ffffff !important;
                    ">
                        POS
                    </div>

                    <div style="
                        padding:12px;
                        background:#4f46e5 !important;
                        color:#ffffff !important;
                    ">
                        Dependency
                    </div>

                    <div style="
                        padding:12px;
                        background:#4f46e5 !important;
                        color:#ffffff !important;
                    ">
                        Head
                    </div>

                </div>

                {dependency_rows}

            </div>

        </div>
        """

    # -------------------------------------------------
    # UNKNOWN OPERATION
    # -------------------------------------------------

    else:

        return """
        <div style="
            padding:20px;
            background:#f8d7da;
            color:#842029;
            border-radius:10px;
        ">
            Invalid analysis operation.
        </div>
        """

In [10]:
def select_operation(name):
    return name, f"### 🟢 Selected Analysis: **{name}**"

In [11]:
import gradio as gr

css = """
body {
    background-color: #f5f7fb !important;
}

.gradio-container {
    background-color: #f5f7fb !important;
    color: #1f2937 !important;
    max-width: 1150px !important;
    margin: 0 auto !important;
    padding: 25px !important;
}

h1 {
    text-align: center;
    color: #1f2937 !important;
}

.subtitle {
    text-align: center;
    color: #6b7280 !important;
    font-size: 17px;
}

.result-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
}

.result-table th {
    background-color: #4f46e5 !important;
    color: white !important;
    padding: 10px;
    text-align: left;
}

.result-table td {
    padding: 9px;
    border-bottom: 1px solid #ddd;
    color: #1f2937 !important;
    background-color: white !important;
}
"""


with gr.Blocks(
    title="NLP Analysis Platform"
) as demo:

    # =========================================
    # HEADER
    # =========================================

    gr.Markdown(
        """
        # 🧠 NLP Analysis Platform

        <div class="subtitle">
        Interactive Natural Language Processing Dashboard
        </div>
        """
    )

    # =========================================
    # INPUT SECTION
    # =========================================

    gr.Markdown("## 📝 Enter Text")

    text_input = gr.Textbox(
        show_label=False,
        placeholder="Type or paste your text here...",
        lines=5
    )

    # =========================================
    # ANALYSIS SELECTION
    # =========================================

    gr.Markdown("## 🔍 Choose Analysis")

    # Hidden state
    operation = gr.State("POS Tagging")

    selected_operation = gr.Markdown(
        "### 🟢 Selected Analysis: **POS Tagging**"
    )

    # =========================================
    # FIRST ROW OF BUTTONS
    # =========================================

    with gr.Row():

        ner_button = gr.Button(
            "🏷️ Named Entities"
        )

        pos_button = gr.Button(
            "🧩 POS Tagging"
        )

        pos_dist_button = gr.Button(
            "📊 POS Distribution"
        )

    # =========================================
    # SECOND ROW OF BUTTONS
    # =========================================

    with gr.Row():

        lemma_button = gr.Button(
            "🔤 Lemmatization"
        )

        stem_button = gr.Button(
            "✂️ Stemming"
        )

        morph_button = gr.Button(
            "🧬 Morphology"
        )

        dep_button = gr.Button(
            "🔗 Dependencies"
        )

    # =========================================
    # BUTTON SELECTION LOGIC
    # =========================================

    ner_button.click(
        fn=lambda: select_operation(
            "Named-Entity Relationship"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    pos_button.click(
        fn=lambda: select_operation(
            "POS Tagging"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    pos_dist_button.click(
        fn=lambda: select_operation(
            "POS Distribution"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    lemma_button.click(
        fn=lambda: select_operation(
            "Lemmatization"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    stem_button.click(
        fn=lambda: select_operation(
            "Stemming"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    morph_button.click(
        fn=lambda: select_operation(
            "Morphology"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    dep_button.click(
        fn=lambda: select_operation(
            "Dependencies"
        ),
        outputs=[
            operation,
            selected_operation
        ]
    )

    # =========================================
    # ANALYZE BUTTON
    # =========================================

    analyze_button = gr.Button(
        "🔍 Analyze Text",
        variant="primary"
    )

    # =========================================
    # RESULT
    # =========================================

    gr.Markdown("## 📊 Result")

    output = gr.HTML()

    # =========================================
    # CONNECT ANALYZE BUTTON
    # =========================================

    analyze_button.click(
        fn=analyze_text,
        inputs=[
            text_input,
            operation
        ],
        outputs=output
    )

    # =========================================
    # FOOTER
    # =========================================

    gr.Markdown(
        """
        ---

        **NLP Analysis Platform**  
        Python • spaCy • NLTK • Gradio
        """
    )


# =============================================
# LAUNCH
# =============================================

demo.launch(css=css)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
